In [0]:
products_data = [
    ("P101", "Laptop", "Electronics", 10),
    ("P102", "Mouse", "Electronics", 20),
    ("P103", "Keyboard", "Electronics", 15),
    ("P104", "Monitor", "Electronics", 8),
    ("P105", "Printer", "Office", 5),
    ("P106", "Chair", "Furniture", 12),
    ("P107", "Desk", "Furniture", 6),
    ("P108", "Router", "Networking", 10),
    ("P109", "Switch", "Networking", 7),
    ("P110", "Scanner", "Office", 5)
]

products_columns = [
    "product_id",
    "product_name",
    "category",
    "reorder_level"
]

products_df = spark.createDataFrame(
    products_data,
    products_columns
)

products_df.show()

+----------+------------+-----------+-------------+
|product_id|product_name|   category|reorder_level|
+----------+------------+-----------+-------------+
|      P101|      Laptop|Electronics|           10|
|      P102|       Mouse|Electronics|           20|
|      P103|    Keyboard|Electronics|           15|
|      P104|     Monitor|Electronics|            8|
|      P105|     Printer|     Office|            5|
|      P106|       Chair|  Furniture|           12|
|      P107|        Desk|  Furniture|            6|
|      P108|      Router| Networking|           10|
|      P109|      Switch| Networking|            7|
|      P110|     Scanner|     Office|            5|
+----------+------------+-----------+-------------+



In [0]:
warehouses_data = [
    ("W1", "Central Warehouse", "Chennai"),
    ("W2", "South Warehouse", "Bangalore"),
    ("W3", "North Warehouse", "Hyderabad")
]

warehouses_columns = [
    "warehouse_id",
    "warehouse_name",
    "city"
]

warehouses_df = spark.createDataFrame(
    warehouses_data,
    warehouses_columns
)

warehouses_df.show()

+------------+-----------------+---------+
|warehouse_id|   warehouse_name|     city|
+------------+-----------------+---------+
|          W1|Central Warehouse|  Chennai|
|          W2|  South Warehouse|Bangalore|
|          W3|  North Warehouse|Hyderabad|
+------------+-----------------+---------+



In [0]:
stock_data = [
    ("M1", "P101", "W1", "IN", 50),
    ("M2", "P102", "W1", "IN", 30),
    ("M3", "P103", "W2", "IN", 25),
    ("M4", "P104", "W2", "IN", 10),
    ("M5", "P105", "W3", "IN", 8),
    ("M6", "P101", "W1", "OUT", 45),
    ("M7", "P102", "W1", "OUT", 5),
    ("M8", "P103", "W2", "OUT", 18),
    ("M9", "P104", "W2", "OUT", 3),
    ("M10", "P105", "W3", "OUT", 6)
]

stock_columns = [
    "movement_id",
    "product_id",
    "warehouse_id",
    "movement_type",
    "quantity"
]

stock_df = spark.createDataFrame(
    stock_data,
    stock_columns
)

stock_df.show()

+-----------+----------+------------+-------------+--------+
|movement_id|product_id|warehouse_id|movement_type|quantity|
+-----------+----------+------------+-------------+--------+
|         M1|      P101|          W1|           IN|      50|
|         M2|      P102|          W1|           IN|      30|
|         M3|      P103|          W2|           IN|      25|
|         M4|      P104|          W2|           IN|      10|
|         M5|      P105|          W3|           IN|       8|
|         M6|      P101|          W1|          OUT|      45|
|         M7|      P102|          W1|          OUT|       5|
|         M8|      P103|          W2|          OUT|      18|
|         M9|      P104|          W2|          OUT|       3|
|        M10|      P105|          W3|          OUT|       6|
+-----------+----------+------------+-------------+--------+



In [0]:
from pyspark.sql.functions import when, col
stock_df = stock_df.withColumn(
    "final_quantity",
    when(
        col("movement_type") == "OUT",
        -col("quantity")
    ).otherwise(col("quantity"))
)
display(stock_df)

movement_id,product_id,warehouse_id,movement_type,quantity,final_quantity
M1,P101,W1,IN,50,50
M2,P102,W1,IN,30,30
M3,P103,W2,IN,25,25
M4,P104,W2,IN,10,10
M5,P105,W3,IN,8,8
M6,P101,W1,OUT,45,-45
M7,P102,W1,OUT,5,-5
M8,P103,W2,OUT,18,-18
M9,P104,W2,OUT,3,-3
M10,P105,W3,OUT,6,-6


In [0]:
stock_summary = stock_df.groupBy(
    "product_id",
    "warehouse_id"
).sum("final_quantity")

In [0]:
stock_summary = stock_summary.withColumnRenamed(
    "sum(final_quantity)",
    "current_stock"
)
display(stock_summary)

product_id,warehouse_id,current_stock
P101,W1,5
P102,W1,25
P103,W2,7
P104,W2,7
P105,W3,2


In [0]:
inventory_df = stock_summary.join(
    products_df,
    "product_id"
    
)
display(inventory_df)

product_id,warehouse_id,current_stock,product_name,category,reorder_level
P101,W1,5,Laptop,Electronics,10
P102,W1,25,Mouse,Electronics,20
P103,W2,7,Keyboard,Electronics,15
P104,W2,7,Monitor,Electronics,8
P105,W3,2,Printer,Office,5


In [0]:
inventory_df = inventory_df.join(
    warehouses_df,
    "warehouse_id"
)
display(inventory_df)

warehouse_id,product_id,current_stock,product_name,category,reorder_level,warehouse_name,warehouse_city
W1,P101,5,Laptop,Electronics,10,Central Warehouse,Chennai
W1,P102,25,Mouse,Electronics,20,Central Warehouse,Chennai
W2,P103,7,Keyboard,Electronics,15,South Warehouse,Bangalore
W2,P104,7,Monitor,Electronics,8,South Warehouse,Bangalore
W3,P105,2,Printer,Office,5,North Warehouse,Hyderabad


In [0]:
inventory_df = inventory_df.withColumn(
    "reorder_flag",
    when(
        col("current_stock") < col("reorder_level"),
        "YES"
    ).otherwise("NO")
)
display(inventory_df)

warehouse_id,product_id,current_stock,product_name,category,reorder_level,warehouse_name,warehouse_city,reorder_flag
W1,P101,5,Laptop,Electronics,10,Central Warehouse,Chennai,YES
W1,P102,25,Mouse,Electronics,20,Central Warehouse,Chennai,NO
W2,P103,7,Keyboard,Electronics,15,South Warehouse,Bangalore,YES
W2,P104,7,Monitor,Electronics,8,South Warehouse,Bangalore,YES
W3,P105,2,Printer,Office,5,North Warehouse,Hyderabad,YES


In [0]:
inventory_df.createOrReplaceTempView(
    "master_inventory_view"
)

In [0]:
%sql
SELECT *
FROM master_inventory_view;

warehouse_id,product_id,current_stock,product_name,category,reorder_level,warehouse_name,warehouse_city,reorder_flag
W1,P101,5,Laptop,Electronics,10,Central Warehouse,Chennai,YES
W1,P102,25,Mouse,Electronics,20,Central Warehouse,Chennai,NO
W2,P103,7,Keyboard,Electronics,15,South Warehouse,Bangalore,YES
W2,P104,7,Monitor,Electronics,8,South Warehouse,Bangalore,YES
W3,P105,2,Printer,Office,5,North Warehouse,Hyderabad,YES


In [0]:
%sql
SELECT *
FROM master_inventory_view
WHERE reorder_flag = 'YES';

warehouse_id,product_id,current_stock,product_name,category,reorder_level,warehouse_name,warehouse_city,reorder_flag
W1,P101,5,Laptop,Electronics,10,Central Warehouse,Chennai,YES
W2,P103,7,Keyboard,Electronics,15,South Warehouse,Bangalore,YES
W2,P104,7,Monitor,Electronics,8,South Warehouse,Bangalore,YES
W3,P105,2,Printer,Office,5,North Warehouse,Hyderabad,YES


In [0]:
warehouses_df = warehouses_df.withColumnRenamed(
    "city",
    "warehouse_city"
)

In [0]:
inventory_df.write.format("delta").mode(
    "overwrite"
).saveAsTable("inventory_delta_table")